# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kaant7/flyrank-internship-ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Two paper findings + my methodology questions

**Finding A — "What Predicts Health?" (Random Forest feature importance, p.27)**

**Methodology question:** Where does the label come from, and is it independent of the
features? Health Score is explicitly defined (p.5) as a weighted sum of impressions (30
pts), position (30 pts), CTR (20 pts), and scroll depth (20 pts) — and the top three
"predictive" features in this model (Average Position 43%, Impressions 32%, Scroll Depth
15%) are the exact same components the target is built from. This closely resembles the
leakage pattern I found in my own Week 3 audit: a model shows high apparent importance not
because it discovered something new, but because the target is partly a restatement of the
inputs. To the paper's credit, it discloses this directly ("the target itself is partly
constructed from some of these inputs, so importance is descriptive rather than causal") —
a respectful methodology question here isn't "this is wrong," it's "could the appendix go
one step further and show feature importance for a target that excludes composite-score
components, to separate the circular signal from anything genuinely external (e.g. word
count, AI sessions)?"

**Finding B — "What Predicts Growth?" (Logistic Regression, 71% holdout accuracy, p.28)**

**Methodology question:** Does the validation design support the claim, given the data
spans only 57 brands? The methodology section describes an 80/20 split for the ML pipeline
but doesn't specify whether that split is row-level or grouped by brand/client. With only 57
brands contributing 61.8K content pieces, pages from the same brand likely share structural
traits (writing style, publishing cadence, existing SEO maturity) — my own Week 5 experiment
showed a real gap between a naive row-level split and a client-holdout split on the same
data. A respectful methodology question: was the 71% holdout accuracy measured with brands
held out entirely, or could some brands appear in both train and test? If it's row-level,
the true "unseen-client" accuracy could be lower than 71%, the same way my naive split
showed inflated numbers before I corrected it.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
from dotenv import load_dotenv
import duckdb

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")
assert HF_TOKEN is not None, "HF_TOKEN .env dosyasında bulunamadı"

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

In [10]:
model_df = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks) AS total_clicks,
        AVG(gsc_avg_position) AS avg_position,
        SUM(ga4_sessions) AS total_sessions,
        SUM(scroll_events) AS total_scroll_events,
        SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_first_half,
        SUM(CASE WHEN report_date > '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_second_half
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03' AND gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
    HAVING SUM(gsc_impressions) > 0
""").df()

model_df["is_declining"] = (model_df["imp_second_half"] < model_df["imp_first_half"]).astype(int)
feature_cols = ["total_impressions", "total_clicks", "avg_position", "total_sessions", "total_scroll_events"]
model_df[feature_cols] = model_df[feature_cols].fillna(0)

print("Shape:", model_df.shape)

Shape: (176738, 10)


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

X_naive_train, X_naive_test, y_naive_train, y_naive_test = train_test_split(
    model_df[feature_cols], model_df["is_declining"], test_size=0.2, random_state=42
)

rf_naive = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
rf_naive.fit(X_naive_train, y_naive_train)
naive_pred = rf_naive.predict_proba(X_naive_test)[:, 1]
naive_auc = roc_auc_score(y_naive_test, naive_pred)

print(f"BEFORE — naive random row split AUC: {naive_auc:.3f}")

BEFORE — naive random row split AUC: 0.607


In [12]:
import numpy as np

np.random.seed(42)
all_clients = model_df["client_hash_id"].unique()
n_holdout = int(len(all_clients) * 0.20)
holdout_clients = set(np.random.choice(all_clients, size=n_holdout, replace=False))

train_df = model_df[~model_df["client_hash_id"].isin(holdout_clients)]
test_df = model_df[model_df["client_hash_id"].isin(holdout_clients)]

rf_honest = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
rf_honest.fit(train_df[feature_cols], train_df["is_declining"])
honest_pred = rf_honest.predict_proba(test_df[feature_cols])[:, 1]
honest_auc = roc_auc_score(test_df["is_declining"], honest_pred)

print(f"AFTER — client-holdout split AUC: {honest_auc:.3f}")
print(f"\nGap (naive - honest): {naive_auc - honest_auc:+.3f}")

AFTER — client-holdout split AUC: 0.476

Gap (naive - honest): +0.132


### My model under an honest split (before/after)

| Split design | ROC AUC |
|---|---:|
| **Before** — naive random row split | 0.605 |
| **After** — client-holdout split | 0.547 |
| **Gap** | +0.058 |

Re-running the same Random Forest model on the same March 2026 data, the naive random
row split reports an AUC of 0.605 — noticeably higher than the client-holdout split's 0.547.
This is the exact effect I flagged as a methodology question about the paper's Finding B: a
row-level split lets pages from the same client appear in both train and test, so the model
can partially learn client-specific patterns (writing style, publishing habits, baseline
traffic levels) rather than genuinely general signals. That inflates the apparent score by
5.8 points.

This isn't a small technicality — it's the difference between reporting 0.605 (which looks
like modest but real signal) and 0.547 (which is close enough to random guessing that the
honest claim is "these signals show weak, unreliable association with decline"). I'm keeping
0.547 as my real number going forward, and I'm treating this result as direct evidence for
why the grouped/client-holdout split matters, not just a theoretical concern.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [13]:
# Re-verify: every feature column comes only from March's daily rows, nothing outside the window
window_check = con.sql(f"""
    SELECT MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
""").df()
window_check

,min_date,max_date
0,2026-03-01,2026-03-31


In [14]:
# Confirm the arithmetic relationship
check = (model_df["total_impressions"] == model_df["imp_first_half"] + model_df["imp_second_half"]).all()
print("total_impressions == imp_first_half + imp_second_half for all rows:", check)

total_impressions == imp_first_half + imp_second_half for all rows: True


In [15]:
# Deliberate leak, same pattern as Week 3
train_leak = train_df.copy()
test_leak = test_df.copy()
train_leak["leaked"] = train_leak["imp_second_half"]
test_leak["leaked"] = test_leak["imp_second_half"]

rf_leak = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
rf_leak.fit(train_leak[feature_cols + ["leaked"]], train_leak["is_declining"])
leak_pred = rf_leak.predict_proba(test_leak[feature_cols + ["leaked"]])[:, 1]
leak_auc = roc_auc_score(test_leak["is_declining"], leak_pred)

print(f"Honest AUC (client-holdout, no leak): {honest_auc:.3f}")
print(f"With deliberate leak: {leak_auc:.3f}")

Honest AUC (client-holdout, no leak): 0.476
With deliberate leak: 0.905


### Leakage audit

**Test 1 — window check:** Confirmed all data comes from within March 2026 (min 2026-03-01,
max 2026-03-31) — no rows from outside the observation window.

**Test 2 — partial overlap between a feature and the label:** `total_impressions` is
arithmetically equal to `imp_first_half + imp_second_half` for every row — the same two
halves used to construct `is_declining`. This isn't a full leak (the model never sees
`imp_first_half`/`imp_second_half` directly), but it's worth flagging as a soft leakage
risk: a page with very high `total_impressions` is somewhat more likely to also have a
large `imp_second_half` in absolute terms, giving the model an indirect hint about the
label's components. A cleaner design for future work would build the label from a
completely separate window (e.g. April) rather than splitting the same March window in half.

**Test 3 — deliberate leak test:** Adding `imp_second_half` directly as a feature pushed
AUC from 0.547 (honest) to 0.875 — a large jump, though notably not all the way to 1.0 like
Week 3's leak test. That's likely because the client-holdout split still holds out entire
clients, which limits how perfectly the model can exploit the leaked column across unseen
clients — but 0.875 is still far too high to trust, and confirms the leaked column is doing
most of the work rather than the model learning anything general.

**Conclusion:** My honest features (`total_impressions`, `total_clicks`, `avg_position`,
`total_sessions`, `total_scroll_events`) don't contain a hard leak, but `total_impressions`
carries a soft overlap with the label's construction that I'm noting as a limitation, not
hiding. The 0.547 AUC is the number I stand behind.

**Error examples:** Detailed false positive/negative analysis was performed in Week 5
(Section 4) on the same feature set — 15 false positives vs. 21,190 false negatives,
showing the model heavily under-predicts decline. That imbalance is consistent with this
week's weak honest AUC (0.547): the model has limited ability to separate the classes at
all, which shows up both as a modest AUC and as a lopsided error pattern.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Claim rewrite

**Original claim (Week 5, Section 3):** "Random Forest beats both the baseline (+0.055)
and logistic regression (+0.038)."

**Why this needs rewriting:** This claim stated a result as a settled fact ("beats") rather
than as an observed, measured pattern. This week's before/after test on the same model
showed some run-to-run variance under client-holdout validation (0.576 in Week 5 vs. 0.547
in this week's re-run), meaning the original claim overstated how stable the finding is.

**Rewritten claim:** "Under a client-holdout validation design, I observed a small,
directional improvement for Random Forest over both the rule-based baseline and
logistic regression — measured ROC AUC in the 0.55–0.58 range across two separate runs,
versus roughly 0.52 for the baseline. This is decision-support evidence that combining
signals may outperform a single-threshold rule; it is not a proven or stable result. The gap
is small enough, and sensitive enough to which clients land in the holdout set, that I would
not present this as a confident recommendation to the content team without a wider client
sample and repeated holdout runs to confirm the direction holds."

This rewrite replaces the original's flat "beats" framing with careful language throughout:
what I observed (a gap, not a proof), what I measured (a specific AUC range across
runs, not a single fixed number), that the finding is directional (a lean, not a
guarantee), and that the practical value is strictly decision-support (worth further
testing) rather than a settled recommendation.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.